# KCT app

**What it does**
A clickable app inside Jupyter for the work on one control, without typing code:

| Tab | You click |
|---|---|
| 1 Case | Pick a case in the **Case** list, choose the control details and the test plan, **Make test steps** (a new run). New control: **+ New case** (just a name), or import its SharePoint folder as a ZIP (select the folder > Download) or a folder path |
| 2 Test steps | **Make test steps**, then pick a step to see its pass and fail criteria |
| 3 Evidence | See the evidence, **Add files** + **Upload**, **Show parts for the AI** |
| 4 Assess | **Run assessment**, with a live progress bar |
| 5 Result | Verdicts, traceability matrix, exceptions, why it can be trusted, the PDF report |
| History | Every earlier run: its documents and test steps, the case's evidence, assessments and the result; **Continue with these test steps** uses them again without the AI |

**Why**
The machine in ING has Jupyter but no web page can be opened, so the app runs inside the notebook: no web server, no open port. It uses the same services as the API and the other notebooks.

**How to use**
1. In the first code cell choose `MODE`: `fake` (sample answers after 1-3 seconds, no AI, no cost), `live` (Gemini, settings from `.env`) or `replay` (recorded real answers, see `demo/README.md`).
2. Run all cells (Run All). The app appears below the last cell.
3. Start in tab 1. The case chosen at the top is used by all tabs. After each step a **Next** button takes you to the next tab. **Run all** (top) does tabs 2 to 5 in one click.
4. After changing `MODE`, restart the kernel (Restart) and run all again.

**Needs**: `ipywidgets` (Step 3 of `notebooks/01_setup.ipynb` installs it). The logo comes from `branding/logo.png` when it is there (see `branding/README.md`).

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

MODE = "fake"     # fake | live | replay

# MODE decides, also when the cell runs again without a kernel restart
for key in ("KCT_AI_PROVIDER", "KCT_FAKE_DELAY"):
    os.environ.pop(key, None)
from services.config import load_env_file, settings
load_env_file()
if MODE == "fake":
    os.environ["KCT_AI_PROVIDER"] = "fake"
    os.environ["KCT_FAKE_DELAY"] = "1-3"   # seconds per answer, so the loaders show
elif MODE == "replay":
    os.environ.update(KCT_AI_PROVIDER="replay", KCT_AI_REPLAY=os.environ.get("KCT_AI_REPLAY", "demo"))
elif os.environ.get("KCT_AI_PROVIDER", "gemini") == "fake":
    raise SystemExit('MODE = "live", but .env sets KCT_AI_PROVIDER=fake: set it to gemini in .env')

from services.app import boot
from services.presentation import notebook as show
from services.presentation.app import KctApp

settings.cache_clear()
boot.cache_clear()
kct = boot()
show.quiet()
print("AI:", settings().ai_provider, settings().ai_model)

In [ ]:
KctApp(kct)